In [13]:
def build_salary_pipeline(path_or_df):
    import pandas as pd
    import numpy as np
    import re
    import unicodedata

    try:
        from rapidfuzz import process, fuzz
        _has_fuzz = True
    except Exception:
        _has_fuzz = False

    def _strip_diacritics_keep_case(s):
        s = "" if pd.isna(s) else str(s)
        return unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")

    def _normalize_text(s):
        s = "" if pd.isna(s) else str(s)
        s = s.replace("\u00a0", " ")
        s = re.sub(r"\s+", " ", s).strip()
        return s

    def _job_title_clean(s):
        s = _normalize_text(s)
        if not s:
            return np.nan
        acronyms = {"HR","CEO","AI","BI","CFO","IT","AML","ESG","SQL","ETL","API","UI","UX","PM","KYC","ML","QA","RPA","ERP","CRM","SRE"}
        tokens = re.split(r"(\s+)", s)
        out = []
        for t in tokens:
            if t.isspace():
                out.append(t)
                continue
            core = re.sub(r"[^\wÕÄÖÜõäöüŠŽšž-]", "", t)
            upper_core = core.upper()
            if upper_core in acronyms:
                out.append(upper_core)
            else:
                parts = core.split("-")
                parts = [p[:1].upper() + p[1:].lower() if p else p for p in parts]
                out.append("-".join(parts))
        return "".join(out).strip()

    def _job_title_norm(s):
        s = _normalize_text(s)
        s = s.lower()
        s = re.sub(r"\s+", " ", s).strip()
        return s if s else np.nan

    def _job_title_normtext(s):
        s = _normalize_text(s)
        s = _strip_diacritics_keep_case(s)
        s = re.sub(r"\s+", " ", s).strip()
        return s if s else np.nan

    def _to_num(series):
        return pd.to_numeric(series, errors="coerce")

    def _experience_group(series):
        x = _to_num(series)
        return pd.cut(x, bins=[-1, 5, 10, 20, 10**9], labels=["0-5", "5-10", "10-20", "20+"])

    def _job_category(title_norm, title_normtext):
        t = "" if pd.isna(title_norm) else str(title_norm)
        tx = "" if pd.isna(title_normtext) else str(title_normtext)
        k = f"{t} {tx}".lower()

        rules = [
            ("Risk / Compliance / ESG", ["aml", "kyc", "compliance", "risk", "esg", "audit", "kontroll", "kontroller", "sisekontroll"]),
            ("Legal", ["jurist", "advokaat", "legal", "õigus", "notar"]),
            ("Healthcare", ["arst", "õde", "füsioterapeut", "fusioterapeut", "psühholoog", "psuhholoog", "terapeut", "resident", "hambaarst", "farmatseut"]),
            ("Engineering / IT", ["it ", "it-", "arendaja", "developer", "programmeer", "software", "devops", "sre", "qa", "test", "andmeinsener", "data engineer", "võrgu", "vorgu", "süsteemiadmin", "sysadmin", "pilve", "cloud", "küber", "kyber", "turbe", "security"]),
            ("Data & Analytics", ["analüüt", "analuut", "analytics", "data", "andme", "bi ", "business intelligence", "sql", "python", "tableau", "power bi", "statistik", "scientist", "ml", "machine learning"]),
            ("Finance", ["finants", "raamat", "accountant", "controller", "kontroller", "maksa", "maksu", "treasury", "auditi", "palkade arvest", "palga arvest"]),
            ("HR", ["personal", "hr", "värb", "varb", "recruit", "talent", "people partner"]),
            ("Marketing / Communication", ["turund", "marketing", "kommunik", "communication", "pr", "brand", "bränd", "brandi", "sotsiaal", "social media", "sisuloo", "content", "copywriter"]),
            ("Sales", ["müük", "muuk", "sales", "müüg", "muug", "account manager", "kliendihaldur", "business development", "bdm"]),
            ("Product", ["toote", "product", "product owner", "tooteoman", "product manager"]),
            ("Operations", ["opera", "logistik", "supply", "tarneahel", "procurement", "hange", "ostu", "operations", "back office", "back-office"]),
            ("Education & Research", ["õpet", "opet", "teacher", "teadur", "research", "professor", "lektor"]),
            ("Public Sector", ["ametnik", "ministeer", "riigi", "vall", "linn", "avalik", "keskus", "politsei", "pääste", "paaste"]),
            ("Consulting", ["konsult", "consult", "nõunik", "nounik", "advisor"]),
            ("Technical / Industrial", ["insener", "engineer", "tehnik", "tootmis", "elektr", "mehaanik", "labor", "keevit", "hooldus"]),
            ("Management", ["juht", "manager", "head of", "team lead", "tiimijuht", "direktor", "chief", "lead"]),
        ]

        for cat, keys in rules:
            if any(kw in k for kw in keys):
                return cat

        return "Other"

    def _derive_job_title_base(df_in):
        df_in = df_in.copy()
        if not _has_fuzz:
            df_in["job_title_base"] = df_in["job_title_clean"]
            return df_in

        titles = df_in["job_title_norm"].fillna("").tolist()
        clean_titles = df_in["job_title_clean"].fillna("").tolist()

        canonical = {}
        canonical_clean = {}
        order = []

        for tn, tc in zip(titles, clean_titles):
            if not tn:
                order.append(np.nan)
                continue
            if not canonical:
                canonical[tn] = tn
                canonical_clean[tn] = tc
                order.append(tn)
                continue
            best = process.extractOne(tn, list(canonical.keys()), scorer=fuzz.ratio)
            if best and best[1] >= 92:
                order.append(best[0])
            else:
                canonical[tn] = tn
                canonical_clean[tn] = tc
                order.append(tn)

        df_in["_base_key"] = order
        base_mode = (
            df_in.groupby("_base_key")["job_title_clean"]
            .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else (s.iloc[0] if len(s) else np.nan))
        )
        df_in["job_title_base"] = df_in["_base_key"].map(base_mode)
        df_in.drop(columns=["_base_key"], inplace=True)
        return df_in

    if isinstance(path_or_df, str):
        raw = pd.read_excel(path_or_df, header=1)
    else:
        raw = path_or_df.copy()

    colmap = {
        "Ajatempel": "timestamp",
        "Kui suur on sinu brutopalk?": "salary",
        "Mis on sinu ametinimetus?": "job_title",
        "Mis on sinu haridustase?": "education",
        "Mitu aastat on selles valdkonnas kogemust?": "experience_years",
        "Kui vana sa oled?": "age",
        "Mitu tundi on keskmiselt nädalas koormus?": "weekly_hours",
        "Kus sa elad?": "location",
    }
    raw = raw.rename(columns={c: colmap.get(c, c) for c in raw.columns})

    for c in ["salary", "experience_years", "age", "weekly_hours"]:
        if c in raw.columns:
            raw[c] = _to_num(raw[c])

    if "weekly_hours" in raw.columns and "salary" in raw.columns:
        raw["salary_per_hour"] = raw["salary"] / (raw["weekly_hours"] * 4)

    if "experience_years" in raw.columns:
        raw["experience_group"] = _experience_group(raw["experience_years"])

    if "job_title" in raw.columns:
        raw["job_title_clean"] = raw["job_title"].apply(_job_title_clean)
        raw["job_title_norm"] = raw["job_title"].apply(_job_title_norm)
        raw["job_title_normtext"] = raw["job_title"].apply(_job_title_normtext)
        raw = _derive_job_title_base(raw)
        raw["job_category"] = [
            _job_category(a, b) for a, b in zip(raw["job_title_norm"], raw["job_title_normtext"])
        ]
        top_n = 20
        top_titles = set(raw["job_title_base"].value_counts().head(top_n).index.tolist())
        raw["job_title_top"] = raw["job_title_base"].apply(lambda x: x if x in top_titles else "Other")

    if "salary" in raw.columns:
        q1 = raw["salary"].quantile(0.25)
        q3 = raw["salary"].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        df_no_outliers = raw[(raw["salary"] >= lower) & (raw["salary"] <= upper)].copy()
    else:
        df_no_outliers = raw.copy()

    final_cols = [
        "timestamp","salary","job_title","education","experience_years","age","weekly_hours","location",
        "salary_per_hour","experience_group","job_title_clean","job_title_norm","job_title_base",
        "job_category","job_title_top","job_title_normtext"
    ]
    existing = [c for c in final_cols if c in df_no_outliers.columns]
    return df_no_outliers[existing]